In [0]:
%sql
CREATE CATALOG IF NOT EXISTS crypto_catalog;

btc_data table 

In [0]:
%sql
USE CATALOG crypto_catalog;

CREATE SCHEMA IF NOT EXISTS btc_data;

USE btc_data;

bronze_btc table 

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bronze_btc (
    Datetime TIMESTAMP,
    Open DOUBLE,
    High DOUBLE,
    Low DOUBLE,
    Close DOUBLE,
    Volume DOUBLE
)
USING DELTA;

now fetching the data from yfinance

In [0]:
%pip install yfinance

In [0]:
import yfinance as yf
import pandas as pd

# Fetch BTC data (last 1 day, hourly)
btc = yf.download("BTC-USD", interval="1h", period="30d")

# Reset index to make Datetime a column
btc.reset_index(inplace=True)

# Show data
display(btc)

now inserting to bronze table 

In [0]:
# Step 1: Flatten columns
btc.columns = [col[0] if isinstance(col, tuple) else col for col in btc.columns]

# Step 2: Select required columns
btc = btc[['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume']]

# Step 3: Convert to Spark
spark_df = spark.createDataFrame(btc)

# Step 4: Cast types (VERY IMPORTANT)
from pyspark.sql.functions import col

spark_df = spark_df.select(
    col("Datetime").cast("timestamp"),
    col("Open").cast("double"),
    col("High").cast("double"),
    col("Low").cast("double"),
    col("Close").cast("double"),
    col("Volume").cast("double")
)

# Step 5: Save to Bronze
spark_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("crypto_catalog.btc_data.bronze_btc")

In [0]:
%sql
SELECT * FROM crypto_catalog.btc_data.bronze_btc

silver table

In [0]:
%sql
CREATE OR REPLACE TABLE silver_btc AS
SELECT DISTINCT 
    Datetime,
    Open,
    High,
    Low,
    Close,
    Volume,
    DATE(Datetime) AS trade_date
FROM crypto_catalog.btc_data.bronze_btc
WHERE Close IS NOT NULL;

In [0]:
%sql
SELECT * FROM silver_btc;

Gold table 

In [0]:
%sql
CREATE OR REPLACE TABLE gold_btc AS
SELECT 
    trade_date,
    AVG(Close) AS avg_price,
    MAX(High) AS highest_price,
    MIN(Low) AS lowest_price,
    SUM(Volume) AS total_volume
FROM silver_btc
GROUP BY trade_date;

In [0]:
%sql
SELECT * FROM gold_btc ORDER BY trade_date;

In [0]:
%sql
CREATE OR REPLACE TABLE feature_btc AS
SELECT 
    Datetime,
    trade_date,
    Open,
    High,
    Low,
    Close,
    Volume,

    -- Price Change
    Close - Open AS price_change,

    -- Return %
    (Close - Open) / Open AS return_pct,

    -- Moving Average
    AVG(Close) OVER (
        ORDER BY Datetime 
        ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
    ) AS moving_avg,

    -- Volatility
    (High - Low) AS volatility

FROM silver_btc
WHERE Close IS NOT NULL;

In [0]:
%sql
SELECT * FROM feature_btc;

In [0]:
# ================================
# IMPORTS
# ================================
from pyspark.sql.window import Window
from pyspark.sql.functions import lag
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# ================================
# LOAD DATA
# ================================
df = spark.table("feature_btc")

# ================================
# CREATE LAG FEATURE (IMPORTANT)
# ================================
window = Window.orderBy("Datetime")

df = df.withColumn("prev_close", lag("Close", 1).over(window))
df = df.dropna()

# ================================
# FEATURE VECTOR
# ================================
assembler = VectorAssembler(
    inputCols=["prev_close", "Open", "Volume"],
    outputCol="features"
)

ml_df = assembler.transform(df).select("Datetime", "features", "Close")

# ================================
# TRAIN MODEL
# ================================
lr = LinearRegression(featuresCol="features", labelCol="Close")
model = lr.fit(ml_df)

# ================================
# PREDICTIONS
# ================================
predictions = model.transform(ml_df)

# PRINT OUTPUT
predictions.select("Datetime", "Close", "prediction").show()

# ================================
# CONVERT TO PANDAS
# ================================
pdf = predictions.select("Datetime", "Close", "prediction").toPandas()
pdf = pdf.sort_values("Datetime")

# Ensure datetime format
pdf["Datetime"] = pd.to_datetime(pdf["Datetime"])

# ================================
# FINAL GRAPH (CLEAN X-AXIS)
# ================================
plt.figure(figsize=(12,6))

plt.plot(pdf["Datetime"], pdf["Close"], marker='o', label="Actual Close")
plt.plot(pdf["Datetime"], pdf["prediction"], linestyle='--', label="Predicted Close")

plt.xlabel("Datetime")
plt.ylabel("Close Price")
plt.title("BTC Actual vs Predicted Close Price (Time Series)")

plt.legend()
plt.grid()

# 🔥 Fix datetime axis
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d-%H:%M'))
plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=1))

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [0]:
plt.figure(figsize=(12, 6))
# Take last 100 values (same as your reference)
actual = pdf["Close"].tail(100).values
predicted = pdf["prediction"].tail(100).values

# Plot like reference
plt.plot(actual, label='Actual Next-Hour Price', color='black')
plt.plot(predicted, label='LR Prediction', color='red', linestyle='--')

plt.title("Model Performance: Actual vs Predicted")
plt.legend()

plt.show()

In [0]:
print(btc.tail())

In [0]:
# Create Moving Average (Trend)
btc["ma_20"] = btc["Close"].rolling(window=20).mean()

import plotly.graph_objects as go

# Optional: last 100 points for clarity
btc_last = btc.tail(100)

fig = go.Figure()

# Actual Close Price
fig.add_trace(go.Scatter(
    x=btc_last['Datetime'],
    y=btc_last['Close'],
    mode='lines',
    name='BTC Close Price',
    line=dict(color='blue', width=2)
))

# Trend Line (Moving Average)
fig.add_trace(go.Scatter(
    x=btc_last['Datetime'],
    y=btc_last['ma_20'],
    mode='lines',
    name='Trend (20-period MA)',
    line=dict(color='orange', width=3, dash='dash')
))

fig.update_layout(
    title='BTC Trend Analysis (Close Price + Moving Average)',
    xaxis_title='Time',
    yaxis_title='Price (USD)',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()